# Neck stretch video → 3D pose

Run a short neck-stretch video through the project's pipeline:

1. Read frames from `data/neck_stretch/neck_strech_vd.mp4`
2. RTMPose (via `Pose2D`) on every frame → sequence of COCO-17 2D keypoints
3. MotionBERT-Lite (via `Pose3D` + `Pose3DBuffer`) on a sliding 27-frame window → sequence of H36M-17 3D keypoints
4. Animate the 3D rig
5. Drive `HoldFSM` + `analysis.rules_hold.score_frame` frame-by-frame to detect the hold, then `score_hold` on completion → `HoldAnalysis`
6. Animated 3D rig with live hold-status panel (state, measured tilt, target, progress, drifts)
7. `ThaiCoachLLM.generate(HoldAnalysis, exercise=...)` with concise system prompt + few-shot priming
8. Gemini TTS → speak the feedback

Pair-companion to `squat_video_to_3d.ipynb` — same stages, but the rep-based `SquatFSM` is replaced by the timed-hold `HoldFSM` and the per-rep loop collapses to a single `HoldAnalysis` at completion. See `neck_stretch_single_frame_endtoend.ipynb` for the still-image counterpart.

Runs on Apple Silicon (2D on CPU/ONNX, 3D on MPS). 2D dominates runtime; expect ~15–20 s on a 900-frame clip plus a couple seconds for the 3D pass.


In [ ]:
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

VIDEO_PATH = PROJECT_ROOT / "data" / "neck_stretch" / "neck_strech_vd.mp4"
assert VIDEO_PATH.exists(), f"missing video: {VIDEO_PATH}"

## 1. Open the video

Probe basic properties and show the first frame as a sanity check.


In [ ]:
%matplotlib inline

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "cv2.VideoCapture failed to open the file"

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = float(cap.get(cv2.CAP_PROP_FPS))
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"video: {W}x{H} @ {FPS:.2f} FPS, {N_FRAMES} frames (~{N_FRAMES / FPS:.1f}s)")

ok, first_bgr = cap.read()
assert ok and first_bgr is not None, "failed to read first frame"
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # rewind

plt.figure(figsize=(5, 8))
plt.imshow(cv2.cvtColor(first_bgr, cv2.COLOR_BGR2RGB))
plt.title("frame 0")
plt.axis("off")
plt.show()

## 2. 2D pose on every frame

Same `Pose2D` wrapper as the single-frame notebook (YOLOX-tiny + RTMPose-s, ONNX/CPU). Returns 17 COCO-2D keypoints + per-joint scores per frame.

We collect the full sequence into `kps_2d_seq` and `scores_seq` so stages 3 and 5 can replay it without re-running ONNX.


In [ ]:
from pose2d import Pose2D
from render import SKELETON  # COCO-17 bones

pose2d = Pose2D(device="cpu", mode="lightweight")

cap = cv2.VideoCapture(str(VIDEO_PATH))
kps_2d_seq = np.zeros((N_FRAMES, 17, 2), dtype=np.float32)
scores_seq = np.zeros((N_FRAMES, 17), dtype=np.float32)

t0 = time.time()
for i in range(N_FRAMES):
    ok, frame_bgr = cap.read()
    if not ok:
        # Some containers report more frames than they actually contain; truncate.
        kps_2d_seq = kps_2d_seq[:i]
        scores_seq = scores_seq[:i]
        N_FRAMES = i
        break
    kps, scores = pose2d.infer(frame_bgr)
    kps_2d_seq[i] = kps
    scores_seq[i] = scores
    if (i + 1) % 50 == 0 or i == 0:
        elapsed = time.time() - t0
        print(
            f"  frame {i + 1:>4}/{N_FRAMES}  elapsed {elapsed:5.1f}s  ({(i + 1) / elapsed:4.1f} fps)"
        )
cap.release()

print(
    f"\n2D pass: {time.time() - t0:.1f}s  kps_2d_seq:{kps_2d_seq.shape}  mean score:{float(scores_seq.mean()):.3f}"
)

In [ ]:
# Sanity overlay: COCO-17 skeleton on frame 0.
fig, ax = plt.subplots(figsize=(5, 8))
ax.imshow(cv2.cvtColor(first_bgr, cv2.COLOR_BGR2RGB))

THRESH = 0.3
kps0, sc0 = kps_2d_seq[0], scores_seq[0]
for a, b in SKELETON:
    if sc0[a] < THRESH or sc0[b] < THRESH:
        continue
    ax.plot(
        [kps0[a, 0], kps0[b, 0]], [kps0[a, 1], kps0[b, 1]], color="orange", linewidth=2
    )
vis = sc0 >= THRESH
ax.scatter(
    kps0[vis, 0], kps0[vis, 1], s=30, c="lime", edgecolors="black", linewidths=0.5
)
ax.set_title("RTMPose 2D keypoints — frame 0")
ax.axis("off")
plt.show()

## 3. Lift to 3D with MotionBERT-Lite

`Pose3DBuffer` holds the most recent 27 H36M-17 poses. Each call to `lift(H, W)` runs MotionBERT-Lite and returns the **centre frame** of the current window. Driving it frame-by-frame yields a 3D pose for every frame from index 13 to index `N-14` inclusive — i.e. `N - 26` 3D poses total, lagging the video by 13 frames.

Frames before the buffer is full are skipped (no centre-frame prediction available).


In [ ]:
from pose3d import Pose3D, Pose3DBuffer, coco17_to_h36m17

pose3d = Pose3D()  # auto-selects MPS on Apple Silicon
buf = Pose3DBuffer(pose3d)

kps_3d_seq = []
centre_frame_indices = []  # original video-frame index that each 3D pose corresponds to
half_win = pose3d.window_size // 2  # 13

t0 = time.time()
for i in range(N_FRAMES):
    h36m = coco17_to_h36m17(kps_2d_seq[i], scores_seq[i])
    buf.push(h36m)
    if buf.ready():
        kps_3d = buf.lift(frame_h=H, frame_w=W)
        kps_3d_seq.append(kps_3d)
        centre_frame_indices.append(i - half_win)
kps_3d_seq = np.stack(kps_3d_seq, axis=0)  # (N - 26, 17, 3)
centre_frame_indices = np.array(centre_frame_indices)

print(f"3D pass: {time.time() - t0:.1f}s  kps_3d_seq:{kps_3d_seq.shape}")
print(
    f"first 3D pose corresponds to video frame {centre_frame_indices[0]}, last to {centre_frame_indices[-1]}"
)

## 4. Animate the 3D rig

`matplotlib.animation.FuncAnimation` driving the same H36M-17 bone topology and axis convention as the single-frame notebook (MotionBERT depth → matplotlib Y, image-down Y → matplotlib +Z so the person stands up).

Axis limits are fixed across the whole sequence so the rig doesn't jitter from frame to frame. Playback runs at the video's native FPS, so timing matches the source clip even though the 3D sequence is 26 frames shorter (13-frame lead-in / 13-frame trail-off).


In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Default animation.embed_limit is 20 MB; long sequences exceed that easily.
mpl.rcParams["animation.embed_limit"] = 96

H36M_SKELETON = [
    (0, 1),
    (1, 2),
    (2, 3),  # right leg
    (0, 4),
    (4, 5),
    (5, 6),  # left leg
    (0, 7),
    (7, 8),
    (8, 9),
    (9, 10),  # spine + head
    (8, 11),
    (11, 12),
    (12, 13),  # left arm
    (8, 14),
    (14, 15),
    (15, 16),  # right arm
]

# Remap (x, y, z) → (matplotlib X, Y, Z) for the whole sequence.
X_seq = kps_3d_seq[:, :, 0]
Y_seq = kps_3d_seq[:, :, 2]
Z_seq = -kps_3d_seq[:, :, 1]
T = X_seq.shape[0]

# Fixed equal-aspect cube over the whole sequence so the rig stays anchored.
all_xyz = np.stack([X_seq, Y_seq, Z_seq], axis=-1).reshape(-1, 3)
mins, maxs = all_xyz.min(axis=0), all_xyz.max(axis=0)
mid = (mins + maxs) / 2
half = (maxs - mins).max() / 2

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(mid[2] - half, mid[2] + half)
ax.set_xlabel("X (left-right)")
ax.set_ylabel("Z (depth)")
ax.set_zlabel("Y (up)")
ax.view_init(elev=15, azim=-70)
title = ax.set_title("")

bone_lines = [
    ax.plot([], [], [], color="royalblue", linewidth=2)[0] for _ in H36M_SKELETON
]
joint_scat = ax.scatter(X_seq[0], Y_seq[0], Z_seq[0], c="crimson", s=30)


def update(t):
    Xt, Yt, Zt = X_seq[t], Y_seq[t], Z_seq[t]
    for (a, b), ln in zip(H36M_SKELETON, bone_lines):
        ln.set_data([Xt[a], Xt[b]], [Yt[a], Yt[b]])
        ln.set_3d_properties([Zt[a], Zt[b]])
    joint_scat._offsets3d = (Xt, Yt, Zt)
    title.set_text(
        f"MotionBERT-Lite 3D pose (H36M-17)  —  3D frame {t}/{T - 1}  (video frame {int(centre_frame_indices[t])})"
    )
    return [*bone_lines, joint_scat, title]


ani = FuncAnimation(fig, update, frames=T, interval=1000 / FPS, blit=False)
plt.close(fig)  # suppress the static initial frame; the animation below is the artifact
HTML(ani.to_jshtml())

## 5. Hold detection + scoring

Drive `HoldFSM` frame-by-frame over the 3D-pose sequence. For every video frame that has a centred 3D pose available, we:

1. wrap the keypoints in a `PoseFrame`
2. call `NeckStretchLeft.measure(frame)` → `{"head_lateral_tilt": deg}` (the same call the live loop makes)
3. call `analysis.rules_hold.score_frame(target, measured)` → `(in_target, violations)`
4. push `in_target` into `HoldFSM.update(in_target, ts)`
5. update a per-joint `max_severity_seen` dict so we can build the final precision score

On the FSM's `COMPLETE` transition we build a `HoldAnalysis` via `analysis.rules_hold.score_hold(...)`. If the clip ends mid-hold (no completion) we synthesise the same payload from the FSM's final `in_target_ms` / `drift_count` so the downstream cells still have something to chew on.

Outputs aligned per-video-frame so the next cell can drive the live readout:

- `hold_states_per_frame[i]` — `HoldState` at frame `i` (frames without 3D stay in `IDLE`)
- `tilt_per_frame[i]` — measured `head_lateral_tilt` (NaN if no 3D yet)
- `in_target_per_frame[i]` — bool, or False where 3D is missing
- `progress_per_frame[i]` — `in_target_ms / target_ms` clipped to [0,1]
- `drift_count_per_frame[i]` — running drift count


In [ ]:
from analysis.phases import HoldFSM
from analysis.rules_hold import score_frame, score_hold
from analysis.types import HoldState, PoseFrame
from exercises.neck_stretch import NeckStretchLeft

exercise = NeckStretchLeft()
TARGET_MS = int(exercise.target.hold_seconds * 1000)

# video-frame index -> position in kps_3d_seq (only valid for centred frames 13..N-14)
video_to_3d_idx = {int(v): t for t, v in enumerate(centre_frame_indices)}

hold_states_per_frame: list[HoldState] = [HoldState.IDLE] * N_FRAMES
tilt_per_frame = np.full(N_FRAMES, np.nan, dtype=np.float32)
in_target_per_frame = np.zeros(N_FRAMES, dtype=bool)
progress_per_frame = np.zeros(N_FRAMES, dtype=np.float32)
drift_count_per_frame = np.zeros(N_FRAMES, dtype=np.int32)

max_severity_seen: dict[str, float] = {j.name: 0.0 for j in exercise.target.joints}
_ctx = {"meta": None}  # captured by callback on COMPLETE


def _on_hold(meta: dict) -> None:
    _ctx["meta"] = meta


fsm = HoldFSM(
    target_seconds=exercise.target.hold_seconds,
    on_hold_complete=_on_hold,
)

for i in range(N_FRAMES):
    ts = i / FPS
    if i not in video_to_3d_idx:
        # No 3D yet — leave FSM in IDLE, leave arrays at defaults.
        hold_states_per_frame[i] = fsm.state
        progress_per_frame[i] = min(1.0, fsm.in_target_ms / TARGET_MS)
        drift_count_per_frame[i] = fsm.drift_count
        continue

    kps_3d_i = kps_3d_seq[video_to_3d_idx[i]]
    pf = PoseFrame(
        timestamp=ts,
        keypoints_2d=kps_2d_seq[i],
        scores=scores_seq[i],
        frame_shape=(H, W),
        keypoints_3d=kps_3d_i,
    )
    measured = exercise.measure(pf)
    in_target, violations = score_frame(exercise.target, measured)

    tilt_per_frame[i] = measured["head_lateral_tilt"]
    in_target_per_frame[i] = in_target

    # Track worst per-joint severity for the precision component.
    for v in violations:
        if v.name in max_severity_seen and v.severity > max_severity_seen[v.name]:
            max_severity_seen[v.name] = v.severity

    state = fsm.update(in_target, ts)
    hold_states_per_frame[i] = state
    progress_per_frame[i] = min(1.0, fsm.in_target_ms / TARGET_MS)
    drift_count_per_frame[i] = fsm.drift_count

# Build a HoldAnalysis from the FSM's view of the world.
if _ctx["meta"] is not None:
    completion_state = "COMPLETE"
    meta = _ctx["meta"]
else:
    completion_state = f"PARTIAL (final state {fsm.state.value})"
    meta = {
        "in_target_ms": fsm.in_target_ms,
        "drift_count": fsm.drift_count,
        "completed_ts": (N_FRAMES - 1) / FPS,
    }

hold_analysis = score_hold(
    exercise_name=exercise.name,
    meta=meta,
    target=exercise.target,
    max_severity_seen=max_severity_seen,
)

print(f"hold session: {completion_state}")
print(
    f"  in_target_ms : {hold_analysis.in_target_ms} / {TARGET_MS}  ({100 * hold_analysis.in_target_ms / TARGET_MS:.1f}%)"
)
print(f"  drift_count  : {hold_analysis.drift_count}")
print(f"  worst sev    : {max_severity_seen}")
print(f"\nscore: {hold_analysis.score}/100")
for name, pts in hold_analysis.components.items():
    cap = {"duration": 50, "precision": 30, "stability": 20}[name]
    print(f"  {name:<10} {pts:>3}/{cap}")
print(f"violations ({len(hold_analysis.violations)}):")
for v in hold_analysis.violations:
    print(f"  - {v.name} (sev {v.severity:.2f}): {v.detail_th}")

## 6. Animated 3D pose with live hold status

Same animation as section 4 but with a right-hand readout panel showing the FSM state, measured vs. target tilt, in-target Y/N, accumulated hold progress, and drift count. The progress bar drives off `progress_per_frame[i]` so it pauses on drift (per spec §5.2) and resumes when the user gets back into target. Once the FSM hits `COMPLETE` the panel pins the final values.


In [ ]:
mpl.rcParams["animation.embed_limit"] = 128  # dual-panel + ~900 frames is heavy

X_seq = kps_3d_seq[:, :, 0]
Y_seq = kps_3d_seq[:, :, 2]
Z_seq = -kps_3d_seq[:, :, 1]
T = X_seq.shape[0]
all_xyz = np.stack([X_seq, Y_seq, Z_seq], axis=-1).reshape(-1, 3)
mins, maxs = all_xyz.min(axis=0), all_xyz.max(axis=0)
mid = (mins + maxs) / 2
half = (maxs - mins).max() / 2

STATE_COLOR = {
    "idle": "#808080",
    "entering": "#1f77b4",
    "holding": "#2ca02c",
    "drifted": "#d62728",
    "complete": "#9467bd",
}

fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(1, 2, width_ratios=[1.4, 1.0])
ax = fig.add_subplot(gs[0, 0], projection="3d")
panel = fig.add_subplot(gs[0, 1])
panel.axis("off")

ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(mid[2] - half, mid[2] + half)
ax.set_xlabel("X (left-right)")
ax.set_ylabel("Z (depth)")
ax.set_zlabel("Y (up)")
ax.view_init(elev=15, azim=-70)
title = ax.set_title("")

bone_lines = [
    ax.plot([], [], [], color="royalblue", linewidth=2)[0] for _ in H36M_SKELETON
]
joint_scat = ax.scatter(X_seq[0], Y_seq[0], Z_seq[0], c="crimson", s=30)

# State chip (top of panel) and readout text.
state_chip = panel.text(
    0.02,
    0.94,
    "",
    transform=panel.transAxes,
    ha="left",
    va="top",
    fontsize=18,
    weight="bold",
    color="white",
    bbox=dict(facecolor="#808080", edgecolor="none", pad=8.0, boxstyle="round,pad=0.4"),
)
readout = panel.text(
    0.02,
    0.74,
    "",
    transform=panel.transAxes,
    ha="left",
    va="top",
    family="monospace",
    fontsize=12,
)
# Manual progress bar (background + foreground rectangles in axes coords).
from matplotlib.patches import Rectangle

BAR_X, BAR_Y, BAR_W, BAR_H = 0.02, 0.10, 0.92, 0.06
panel.add_patch(
    Rectangle(
        (BAR_X, BAR_Y),
        BAR_W,
        BAR_H,
        transform=panel.transAxes,
        facecolor="#dddddd",
        edgecolor="#666",
        linewidth=1,
    )
)
bar_fg = Rectangle(
    (BAR_X, BAR_Y),
    0.0,
    BAR_H,
    transform=panel.transAxes,
    facecolor="#2ca02c",
    edgecolor="none",
)
panel.add_patch(bar_fg)
bar_label = panel.text(
    BAR_X + BAR_W / 2,
    BAR_Y + BAR_H / 2,
    "",
    transform=panel.transAxes,
    ha="center",
    va="center",
    fontsize=11,
    color="black",
)

TARGET_DEG = exercise.target.joints[0].target_deg
TOL_DEG = exercise.target.joints[0].tolerance_deg


def _panel_text(video_i: int) -> str:
    tilt = float(tilt_per_frame[video_i])
    in_target = bool(in_target_per_frame[video_i])
    drifts = int(drift_count_per_frame[video_i])
    held_ms = int(progress_per_frame[video_i] * TARGET_MS)
    tilt_str = f"{tilt:+.1f}°" if not np.isnan(tilt) else "--"
    return (
        f"video frame   : {video_i:>4} / {N_FRAMES - 1}\n"
        f"tilt measured : {tilt_str:>8}\n"
        f"tilt target   : {TARGET_DEG:+6.1f}°  ± {TOL_DEG:.0f}°\n"
        f"in-target     : {'YES' if in_target else 'no':>8}\n"
        f"held          : {held_ms:>5} / {TARGET_MS} ms\n"
        f"drifts        : {drifts:>4}"
    )


def update(t):
    Xt, Yt, Zt = X_seq[t], Y_seq[t], Z_seq[t]
    for (a, b), ln in zip(H36M_SKELETON, bone_lines):
        ln.set_data([Xt[a], Xt[b]], [Yt[a], Yt[b]])
        ln.set_3d_properties([Zt[a], Zt[b]])
    joint_scat._offsets3d = (Xt, Yt, Zt)

    video_i = int(centre_frame_indices[t])
    title.set_text(f"3D frame {t}/{T - 1}")

    state = hold_states_per_frame[video_i].value
    state_chip.set_text(state.upper())
    state_chip.get_bbox_patch().set_facecolor(STATE_COLOR.get(state, "#808080"))
    readout.set_text(_panel_text(video_i))

    progress = float(progress_per_frame[video_i])
    bar_fg.set_width(BAR_W * progress)
    bar_label.set_text(f"{int(progress * 100)}%")
    return [*bone_lines, joint_scat, title, state_chip, readout, bar_fg, bar_label]


ani = FuncAnimation(fig, update, frames=T, interval=1000 / FPS, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

## 7. Thai coaching message for the completed hold

Reuses the **concise coach voice** + **two few-shot pairs** from `neck_stretch_single_frame_endtoend.ipynb` so the output matches the target style `ทำได้ดีมาก ทำศีรษะให้นิ่งไว้ ศีรษะไม่เอียงไปทางซ้ายมากเกินไป`. We bypass `ThaiCoachLLM.generate` and feed a custom `messages` list directly to `mlx_vlm.generate`, reusing the loaded model objects on `coach._model` / `coach._processor` / `coach._config`.

Single call (one hold per video), so no rate-limit / caching gymnastics — different from the squat-video notebook's per-rep loop.


In [ ]:
import re

from feedback.llm import ThaiCoachLLM
from feedback.prompt_th import build_hold_summary_prompt

QWEN_DIR = PROJECT_ROOT / "models" / "qwen3_5_4b_mxfp4"
assert QWEN_DIR.exists(), (
    f"missing Qwen weights: {QWEN_DIR} — run scripts/download_models.py"
)

SYSTEM_TH_HOLD_CONCISE = (
    "คุณเป็นโค้ชยืดเหยียดที่พูดสั้น กระชับ เป็นภาษาไทย "
    "ตอบเป็นวลีสั้น ๆ 2-3 วลีในประโยคเดียว เริ่มด้วยคำชม แล้วบอกสิ่งที่ต้องปรับให้ชัดเจน "
    'เลียนสไตล์ตัวอย่างนี้: "ทำได้ดีมาก ทำศีรษะให้นิ่งไว้ ศีรษะไม่เอียงไปทางซ้ายมากเกินไป" '
    'ห้ามใช้ Markdown หัวข้อ หรือบุลเล็ต ห้ามใช้ภาษาอังกฤษ ห้ามตอบเกิน 3 วลี ห้ามขึ้นต้นด้วย "ผล" หรือคำนำใด ๆ'
)

FEW_SHOTS = [
    {
        "user": (
            "ผลการทำท่า ยืดคอด้านซ้าย: คะแนนรวม 95/100\n"
            "  ระยะเวลา: 50/50\n"
            "  ความแม่นยำ: 28/30\n"
            "  ความนิ่ง: 17/20\n"
            "ข้อสังเกต: (ไม่มี)\n"
            "ช่วยสรุปสั้น ๆ เป็นภาษาไทยว่าทำดีตรงไหน ควรปรับตรงไหน"
        ),
        "assistant": "ทำได้ดีมาก คอเอียงเข้าเป้าตลอด ทรงนิ่งสม่ำเสมอจนครบเวลา",
    },
    {
        "user": (
            "ผลการทำท่า ยืดคอด้านซ้าย: คะแนนรวม 68/100\n"
            "  ระยะเวลา: 35/50\n"
            "  ความแม่นยำ: 18/30\n"
            "  ความนิ่ง: 15/20\n"
            "ข้อสังเกต: เอียงศีรษะไปทางซ้ายมากขึ้นอีกนิด (ระดับ 0.60)\n"
            "ช่วยสรุปสั้น ๆ เป็นภาษาไทยว่าทำดีตรงไหน ควรปรับตรงไหน"
        ),
        "assistant": "ตั้งใจดี เอียงคอไปทางซ้ายให้มากขึ้นอีกนิด ทรงท่านิ่งจนครบเวลา",
    },
]


def generate_concise(coach, analysis, exercise, max_tokens: int = 96) -> str:
    from mlx_vlm import generate as mlx_generate
    from mlx_vlm.prompt_utils import apply_chat_template

    user = build_hold_summary_prompt(analysis, exercise)
    messages = [{"role": "system", "content": SYSTEM_TH_HOLD_CONCISE}]
    for ex in FEW_SHOTS:
        messages.append({"role": "user", "content": ex["user"]})
        messages.append({"role": "assistant", "content": ex["assistant"]})
    messages.append({"role": "user", "content": user})

    prompt = apply_chat_template(
        coach._processor, coach._config, messages, num_images=0, enable_thinking=False
    )
    result = mlx_generate(
        coach._model,
        coach._processor,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=False,
    )
    text = getattr(result, "text", str(result))
    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()


t0 = time.time()
coach = ThaiCoachLLM(model_dir=QWEN_DIR)
coach.warmup()
print(f"loaded + warmup in {time.time() - t0:.1f}s\n")

t0 = time.time()
coach_text = generate_concise(coach, hold_analysis, exercise)
print(f"generated in {time.time() - t0:.1f}s\n")
print(
    f"=== Hold  (score {hold_analysis.score}/100, in_target {hold_analysis.in_target_ms}/{TARGET_MS} ms, drifts {hold_analysis.drift_count}) ==="
)
print(f"→ {coach_text}")

## 8. Speak the feedback — Gemini TTS

Single TTS call — one hold, one message, no preamble. Same call pattern as the single-frame notebook (Gemini 2.5 Flash TTS, `Charon` voice, 24 kHz mono PCM in a WAV container). Written to `notebooks/data/neck_stretch_vd_coach.wav` so it doesn't clobber the still-image notebook's output.

Reads `google_ai_studio_api_key` from `.env` — matching the existing convention in `scripts/test_sound.py`.


In [ ]:
import os
import wave

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Audio

load_dotenv(PROJECT_ROOT / ".env")
tts_client = genai.Client(api_key=os.environ["google_ai_studio_api_key"])

# Speak the concise coach line verbatim — no preamble, so the audio matches the on-screen text 1:1.
tts_script = coach_text
print(f"--- tts_script ({len(tts_script)} chars) ---")
print(repr(tts_script))

t0 = time.time()
tts_response = tts_client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=tts_script,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Charon")
            )
        ),
    ),
)
print(f"\nTTS call returned in {time.time() - t0:.1f}s")

if not tts_response.candidates:
    raise RuntimeError("TTS returned no candidates")
cand = tts_response.candidates[0]
if cand.content is None or not getattr(cand.content, "parts", None):
    raise RuntimeError(
        f"TTS produced no audio content. "
        f"finish_reason={getattr(cand, 'finish_reason', None)} "
        f"safety_ratings={getattr(cand, 'safety_ratings', None)}"
    )
part = cand.content.parts[0]
if part.inline_data is None:
    raise RuntimeError("TTS part has no inline_data")

pcm = part.inline_data.data
wav_path = PROJECT_ROOT / "notebooks" / "data" / "neck_stretch_vd_coach.wav"
wav_path.parent.mkdir(parents=True, exist_ok=True)
with wave.open(str(wav_path), "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(pcm)
print(f"wrote {len(pcm)} bytes of PCM → {wav_path.relative_to(PROJECT_ROOT)}\n")

Audio(str(wav_path), autoplay=False)